In [1]:

%pip install -r ../requirements.txt

# from src.preprocessing import AudioPreprocessor
# from src.data_augmentation import DataAugmentation
from src.augmented_dataset_generator import build_dataset
# from src.models.mlp import MLP, train_mlp
# from src.data_loader import get_kfold_dataloaders


import kagglehub
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import torch

import time
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, confusion_matrix, classification_report

import os
import IPython.display as ipd

import shutil
import time
import keras

from src.helper_functions.run_experiement import run_experiment
from src.models.wav2vec2 import run_wav2vec2_experiment, get_device




print("Custom modules loaded successfully!")






[notice] A new release of pip is available: 23.2.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Custom modules loaded successfully!


In [ ]:
import tensorflow as tf

gpu_list = tf.config.list_physical_devices('GPU')

if len(gpu_list) > 0:
    print(f"SUCCESS! GPU is active: {gpu_list[0]}")
else:
    # If this says 0, check macOS version
    print("Still not seeing it. We might need a system-level check.")

In [ ]:
get_device()

In [ ]:
path_to_raw_ravdess_dataset = "./data/raw_ravdess_audio_files"

# Download latest version
downloaded_path = kagglehub.dataset_download("uwrfkaggler/ravdess-emotional-speech-audio")

print("Path to dataset files:", downloaded_path)

if not os.path.exists(path_to_raw_ravdess_dataset):
    shutil.move(downloaded_path, path_to_raw_ravdess_dataset)
    print(f"Success, raw ravdess dataset is now in: {path_to_raw_ravdess_dataset}")
else:
    print("Folder already exists, skipping.")


In [ ]:

NPY_DIR = "./data/processed_spectrograms"
WAV_DIR = "./data/augmented_audio_files"

# Pass the 'path' from kagglehub straight into your script
build_dataset(path_to_raw_ravdess_dataset, NPY_DIR, WAV_DIR)

In [ ]:


# --- HARDCODED TARGET ---
base_name = "03-01-08-02-02-02-24"
processed_dir = "data/processed_spectrograms"
aug_wav_dir = "data/augmented_audio_files"

# 1. Load the Spectrogram Arrays
orig_npy = os.path.join(processed_dir, f"orig_{base_name}.npy")
aug_npy = os.path.join(processed_dir, f"aug_{base_name}.npy")

# 2. Plotting the Spectrograms Side-by-Side
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

if os.path.exists(orig_npy) and os.path.exists(aug_npy):
    # Original Plot
    axes[0].imshow(np.load(orig_npy), aspect='auto', origin='lower', cmap='magma')
    axes[0].set_title(f"ORIGINAL SPECTROGRAM\n({base_name})")
    
    # Augmented Plot
    axes[1].imshow(np.load(aug_npy), aspect='auto', origin='lower', cmap='magma')
    axes[1].set_title(f"AUGMENTED SPECTROGRAM\n(0.5 Traffic + Masking)")
else:
    print("Check your paths! The .npy files weren't found.")

plt.tight_layout()
plt.show()

# 3. Audio Players
# If you don't have a raw folder, we skip the 'original' audio player 
# and just show the augmented one that YOU created.
aug_wav_path = os.path.join(aug_wav_dir, f"aug_{base_name}.wav")

if os.path.exists(aug_wav_path):
    print("🔊 LISTEN TO THE AUGMENTED OUTPUT:")
    ipd.display(ipd.Audio(aug_wav_path))
else:
    print(f"Could not find: {aug_wav_path}")

In [ ]:
# # running MLP Model
# with tf.device("/GPU:0"):
#     mlp_test_acc, mlp_val_accuracies = run_experiment(model_type='mlp')

In [ ]:
# running CNN-LSTM Model

# with tf.device("/GPU:0"):
#     cnn_lstm_mean_val_accuracies, cnn_lstm_val_accuracies = run_experiment(model_type='cnn_lstm', validation_only=False) 

In [2]:
wav2vec2_results = run_wav2vec2_experiment(
    max_folds=None,
    epochs=15,
    batch_size=4,
    learning_rate=1e-5,
    freeze_encoder=True,
    unfreeze_last_n_layers=4,
    early_stopping_patience=3,
    evaluate_test=True,
    save_best_model=True,
)

wav2vec2_results



STARTING WAV2VEC2 FOLD 1/5


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
projector.weight             | MISSING    | 
classifier.bias              | MISSING    | 
projector.bias               | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 01/15: Train Acc: 0.2139 | Val Acc: 0.2987 | Loss: 2.0044 | Val Loss: 1.8664 | Val F1: 0.1971
Epoch 02/15: Train Acc: 0.3648 | Val Acc: 0.3810 | Loss: 1.7444 | Val Loss: 1.6882 | Val F1: 0.3418
Epoch 03/15: Train Acc: 0.5081 | Val Acc: 0.5108 | Loss: 1.4798 | Val Loss: 1.4004 | Val F1: 0.4665
Epoch 04/15: Train Acc: 0.6080 | Val Acc: 0.6234 | Loss: 1.2271 | Val Loss: 1.1870 | Val F1: 0.5800
Epoch 05/15: Train Acc: 0.7123 | Val Acc: 0.7100 | Loss: 0.9832 | Val Loss: 0.9589 | Val F1: 0.6874
Epoch 06/15: Train Acc: 0.7752 | Val Acc: 0.7186 | Loss: 0.8005 | Val Loss: 0.7881 | Val F1: 0.7071
Epoch 07/15: Train Acc: 0.8306 | Val Acc: 0.7706 | Loss: 0.6503 | Val Loss: 0.6648 | Val F1: 0.7633
Epoch 08/15: Train Acc: 0.8686 | Val Acc: 0.7879 | Loss: 0.5033 | Val Loss: 0.6107 | Val F1: 0.7830
Epoch 09/15: Train Acc: 0.8838 | Val Acc: 0.7879 | Loss: 0.4236 | Val Loss: 0.6116 | Val F1: 0.7861
Epoch 10/15: Train Acc: 0.9121 | Val Acc: 0.8355 | Loss: 0.3519 | Val Loss: 0.5464 | Val F1: 0.8384


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

...

[transformers] Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
projector.weight             | MISSING    | 
classifier.bias              | MISSING    | 
projector.bias               | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 01/15: Train Acc: 0.2126 | Val Acc: 0.2957 | Loss: 2.0160 | Val Loss: 1.8552 | Val F1: 0.1775
Epoch 02/15: Train Acc: 0.3341 | Val Acc: 0.3696 | Loss: 1.7880 | Val Loss: 1.6381 | Val F1: 0.2981
Epoch 03/15: Train Acc: 0.4707 | Val Acc: 0.5174 | Loss: 1.5672 | Val Loss: 1.3874 | Val F1: 0.4468
Epoch 04/15: Train Acc: 0.5835 | Val Acc: 0.5826 | Loss: 1.3132 | Val Loss: 1.2030 | Val F1: 0.5479
Epoch 05/15: Train Acc: 0.7234 | Val Acc: 0.6826 | Loss: 1.0378 | Val Loss: 0.9310 | Val F1: 0.6570
Epoch 06/15: Train Acc: 0.7755 | Val Acc: 0.7217 | Loss: 0.8273 | Val Loss: 0.8024 | Val F1: 0.7079
Epoch 07/15: Train Acc: 0.8514 | Val Acc: 0.7826 | Loss: 0.6424 | Val Loss: 0.6686 | Val F1: 0.7773
Epoch 08/15: Train Acc: 0.8807 | Val Acc: 0.8000 | Loss: 0.5169 | Val Loss: 0.5510 | Val F1: 0.7984
Epoch 09/15: Train Acc: 0.9208 | Val Acc: 0.8087 | Loss: 0.3828 | Val Loss: 0.5331 | Val F1: 0.8058
Epoch 10/15: Train Acc: 0.9208 | Val Acc: 0.7870 | Loss: 0.3481 | Val Loss: 0.5339 | Val F1: 0.7854


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


WAV2VEC2 VALIDATION SUMMARY
Folds ran: 5
Best fold: 4
Mean Best Val Acc: 0.8620
Std Best Val Acc: 0.0161
Held-out Test Acc: 0.8438
Held-out Test F1: 0.8431
Saved best Wav2Vec2 model to /Users/manumalakannavar/Documents/VividaFinal/ml_engine/src/models/../../results/wav2vec2_best


,fold,best_epoch,best_val_accuracy,best_val_weighted_precision,best_val_weighted_recall,best_val_weighted_f1,final_val_accuracy,confusion_matrix_path
0,1,10,0.835498,0.849519,0.835498,0.838421,0.818182,/Users/manumalakannavar/Documents/VividaFinal/...
1,2,15,0.857143,0.868581,0.857143,0.858796,0.857143,/Users/manumalakannavar/Documents/VividaFinal/...
2,3,14,0.873913,0.883883,0.873913,0.875317,0.860870,/Users/manumalakannavar/Documents/VividaFinal/...
3,4,14,0.882609,0.890849,0.882609,0.882350,0.865217,/Users/manumalakannavar/Documents/VividaFinal/...
4,5,15,0.860870,0.865950,0.860870,0.860326,0.860870,/Users/manumalakannavar/Documents/VividaFinal/...
